# Лабораторная работа №4. Часть 2. Обучение и подбор гиперпараметров

В ноутбуке строятся единые конвейеры предобработки и четыре классификатора: kNN, дерево решений, линейный SVM (`LinearSVC`) и многослойный персептрон (`MLPClassifier`). Для каждой модели используется `RandomizedSearchCV` с кросс-валидацией. Лучшие модели и тестовые выборки сохраняются в `part3/models/classification_models.joblib` для части 3 и Streamlit.

Чтобы вычисления на датасете smoking оставались выполнимыми на обычном компьютере, используется стратифицированная случайная выборка до 10 000 строк. Drug200 используется полностью.

In [ ]:
from pathlib import Path
import warnings
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

RANDOM_STATE = 42
BASE_DIR = Path.cwd()
if (BASE_DIR / "labs" / "lab4").exists(): LAB4_DIR = BASE_DIR / "labs" / "lab4"
elif (BASE_DIR / "lab4").exists(): LAB4_DIR = BASE_DIR / "lab4"
else: LAB4_DIR = Path("../")
DATA_DIR = LAB4_DIR / "part1" / "datasets"
MODELS_DIR = LAB4_DIR / "part3" / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore", category=UserWarning)

## 1. Подготовка общего конвейера

Предобработка находится внутри `Pipeline`, поэтому на каждом фолде кросс-валидации импутеры, кодировщик и масштабирование обучаются только на соответствующей обучающей части. Это предотвращает утечку информации из validation/test.

In [ ]:
def make_preprocessor(X):
    numeric = X.select_dtypes(include=["number"]).columns.tolist()
    categorical = X.select_dtypes(exclude=["number"]).columns.tolist()
    numeric_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
    categorical_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OneHotEncoder(handle_unknown="ignore"))])
    return ColumnTransformer([("num", numeric_pipe, numeric), ("cat", categorical_pipe, categorical)])

model_spaces = {
    "kNN": (KNeighborsClassifier(), {"clf__n_neighbors": [3, 5, 7, 11], "clf__weights": ["uniform", "distance"], "clf__p": [1, 2]}),
    "Decision Tree": (DecisionTreeClassifier(random_state=RANDOM_STATE), {"clf__max_depth": [None, 4, 8, 12, 20], "clf__min_samples_split": [2, 5, 10], "clf__criterion": ["gini", "entropy"]}),
    "SVM (LinearSVC)": (LinearSVC(random_state=RANDOM_STATE, max_iter=5000), {"clf__C": [0.01, 0.1, 1.0, 10.0], "clf__class_weight": [None, "balanced"]}),
    "MLP": (MLPClassifier(random_state=RANDOM_STATE, max_iter=250, early_stopping=True), {"clf__hidden_layer_sizes": [(50,), (100,), (50, 25)], "clf__alpha": [0.0001, 0.001, 0.01], "clf__learning_rate_init": [0.001, 0.01]})
}
print("Алгоритмы:", list(model_spaces))

## 2. Функция обучения и оптимизации

Для сравнимости все алгоритмы используют одно и то же train/test-разбиение внутри каждого датасета. Основная метрика выбора — macro-F1: она одинаково учитывает каждый класс и полезна при возможном дисбалансе. Дополнительно сохраняются accuracy, weighted precision/recall/F1. Для небольшого Drug200 применяется 3-fold CV; для smoking тоже 3-fold CV.

In [ ]:
def prepare_dataset(name):
    if name == "Smoking":
        df = pd.read_csv(DATA_DIR / "smoking.csv")
        if len(df) > 10000:
            df, _ = train_test_split(df, train_size=10000, random_state=RANDOM_STATE, stratify=df["smoking"])
        X = df.drop(columns=["smoking", "ID"], errors="ignore")
        y = df["smoking"].astype(int)
    else:
        df = pd.read_csv(DATA_DIR / "drug200.csv")
        X = df.drop(columns=["Drug"])
        y = df["Drug"].astype(str)
    return train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

def train_dataset(name, X_train, X_test, y_train, y_test):
    results = {}
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    for model_name, (estimator, params) in model_spaces.items():
        print(f"\n{name}: {model_name}")
        pipe = Pipeline([("preprocessor", make_preprocessor(X_train)), ("clf", estimator)])
        search = RandomizedSearchCV(pipe, params, n_iter=4, scoring="f1_macro", cv=cv,
                                    random_state=RANDOM_STATE, n_jobs=-1, refit=True, error_score="raise")
        search.fit(X_train, y_train)
        best = search.best_estimator_
        pred = best.predict(X_test)
        results[model_name] = {
            "model": best, "best_params": search.best_params_,
            "cv_macro_f1": float(search.best_score_),
            "metrics": {"Accuracy": float(accuracy_score(y_test, pred)),
                        "Precision_macro": float(precision_score(y_test, pred, average="macro", zero_division=0)),
                        "Recall_macro": float(recall_score(y_test, pred, average="macro", zero_division=0)),
                        "F1_macro": float(f1_score(y_test, pred, average="macro", zero_division=0))}
        }
        print("Лучшие параметры:", search.best_params_)
        print("Macro-F1 CV:", round(search.best_score_, 4))
    return results

all_results = {}
for dataset_name in ["Smoking", "Drug"]:
    X_train, X_test, y_train, y_test = prepare_dataset(dataset_name)
    models = train_dataset(dataset_name, X_train, X_test, y_train, y_test)
    all_results[dataset_name] = {"models": models, "X_train": X_train, "X_test": X_test,
                                 "y_train": y_train, "y_test": y_test,
                                 "target_names": sorted(pd.Series(y_train).astype(str).unique().tolist())}
    print(f"\n{dataset_name} завершён: train={len(X_train)}, test={len(X_test)}")

bundle_path = MODELS_DIR / "classification_models.joblib"
joblib.dump(all_results, bundle_path, compress=3)
print("\nАртефакты сохранены:", bundle_path.resolve())

## 3. Сравнение результатов на тестовой выборке

Эта таблица — первичная проверка. Полный classification report и confusion matrix будут построены в части 3.

In [ ]:
rows = []
for dataset_name, data in all_results.items():
    for model_name, item in data["models"].items():
        rows.append({"Датасет": dataset_name, "Модель": model_name, "CV Macro-F1": item["cv_macro_f1"], **item["metrics"], "Лучшие параметры": str(item["best_params"])})
comparison = pd.DataFrame(rows).sort_values(["Датасет", "F1_macro"], ascending=[True, False])
display(comparison)